# Part I. ETL Pipeline for Pre-Processing the Files

## PLEASE RUN THE FOLLOWING CODE FOR PRE-PROCESSING THE FILES

#### Import Python packages 

In [1]:
# Import Python packages 
import pandas as pd
import cassandra
import os
import glob
import csv

In [ ]:
filepath = os.getcwd() + '/event_data'
print(filepath)

#### Creating list of filepaths to process original event csv data files

In [ ]:
# checking your current working directory
print(os.getcwd())

# Get your current folder and subfolder event data
filepath = os.getcwd() + '/event_data'

# Create a for loop to create a list of files and collect each filepath
for root, dirs, files in os.walk(filepath):
# join the file path and roots with the subdirectories using glob
    file_path_list = glob.glob(os.path.join(root,'*.csv'))

#### Processing the files to create the data file csv that will be used for Apache Casssandra tables

In [ ]:
# initiating an empty list of rows that will be generated from each file
full_data_rows_list = [] 
    
# for every filepath in the file path list 
for f in file_path_list:

# reading csv file 
    with open(f, 'r', encoding = 'utf8', newline='') as csvfile: 
        # creating a csv reader object 
        csvreader = csv.reader(csvfile) 
        next(csvreader)
        
 # extracting each data row one by one and append it        
        for line in csvreader:
            full_data_rows_list.append(line) 
            

# creating a smaller event data csv file called event_datafile_full csv that will be used to insert data into the \
# Apache Cassandra tables
csv.register_dialect('myDialect', quoting=csv.QUOTE_ALL, skipinitialspace=True)

with open('event_datafile_new.csv', 'w', encoding = 'utf8', newline='') as f:
    writer = csv.writer(f, dialect='myDialect')
    writer.writerow(['artist','firstName','gender','itemInSession','lastName','length',\
                'level','location','sessionId','song','userId'])
    for row in full_data_rows_list:
        if (row[0] == ''):
            continue
        writer.writerow((row[0], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[12], row[13], row[16]))


In [ ]:
# check the number of rows in your csv file
with open('event_datafile_new.csv', 'r', encoding = 'utf8') as f:
    print(sum(1 for line in f))

# Part II. Complete the Apache Cassandra coding portion of your project. 

## Now you are ready to work with the CSV file titled <font color=red>event_datafile_new.csv</font>, located within the Workspace directory.  The event_datafile_new.csv contains the following columns: 
- artist 
- firstName of user
- gender of user
- item number in session
- last name of user
- length of the song
- level (paid or free song)
- location of the user
- sessionId
- song title
- userId

The image below is a screenshot of what the denormalized data should appear like in the <font color=red>**event_datafile_new.csv**</font> after the code above is run:<br>

<img src="images/image_event_datafile_new.jpg">

## Begin writing your Apache Cassandra code in the cells below

#### Creating a Cluster

In [ ]:
# This should make a connection to a Cassandra instance your local machine 
# (127.0.0.1)

from cassandra.cluster import Cluster
try:
    cluster = Cluster(['127.0.0.1'])
    # To establish connection and begin executing queries, need a session
    session = cluster.connect()
except Exception as e:
    print(e)

#### Create Keyspace

In [ ]:
# TO-DO: Create a Keyspace
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS sparkify 
    WITH REPLICATION = 
    { 'class' : 'SimpleStrategy', 'replication_factor' : 1 }"""
)
except Exception as e:
    print(e)


#### Set Keyspace

In [ ]:
# TO-DO: Set KEYSPACE to the keyspace specified above
try:
    session.set_keyspace('sparkify')
except Exception as e:
    print(e)


### Now we need to create tables to run the following queries. Remember, with Apache Cassandra you model the database tables on the queries you want to run.

## Create queries to ask the following three questions of the data

### 1. Give me the artist, song title and song's length in the music app history that was heard during  sessionId = 338, and itemInSession  = 4


### 2. Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name) for userid = 10, sessionid = 182
    

### 3. Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'




In [ ]:
df_data = pd.read_csv('event_datafile_new.csv')
# Check if the combination of 'song' and 'userId' is always unique
is_unique = not df_data.duplicated(subset=['song', 'userId']).any()
print(f"Is the combination of 'song' and 'userId' always unique? {is_unique}")

# If you want to see the duplicates (if any), you can use:
duplicates = df_data[df_data.duplicated(subset=['song', 'userId'], keep=False)]
duplicates

In [ ]:
df_data[(df_data['song'] == "OMG") & (df_data['userId'] == 80)]

In [ ]:
## Query 1:  Give me the artist, song title and song's length in the music app history that was heard during \
## sessionId = 338, and itemInSession = 4
query = """
    CREATE TABLE IF NOT EXISTS music_session_item (
        sessionId int,
        itemInSession int,
        artist text,
        song text,
        length float,
        PRIMARY KEY (sessionId, itemInSession)
    );
"""

try:
    session.execute(query)
except Exception as e:
    print(e)                    

The primary key for the `music_session_item` table is defined as `(sessionId, itemInSession)`. This design is chosen to efficiently support the query:

> "Give me the artist, song title and song's length in the music app history that was heard during sessionId = 338, and itemInSession = 4"

- **Partition Key (`sessionId`)**: Groups all events for a specific session together, allowing fast access to all items in a session.
- **Clustering Key (`itemInSession`)**: Orders the events within a session by the sequence they occurred, enabling direct lookup of a specific item in a session.

This composite key ensures each row is uniquely identified by its session and item sequence, and allows efficient retrieval for queries filtering by both `sessionId` and `itemInSession`.

In [ ]:
# We have provided part of the code to set up the CSV file. Please complete the Apache Cassandra code below#
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
## Assign the INSERT statements into the `query` variable
        query = "INSERT INTO music_session_item (sessionId, itemInSession, artist, song, length) VALUES (%s, %s, %s, %s, %s)"

        ## Assign which column element should be assigned for each column in the INSERT statement.
        ## For e.g., to INSERT artist_name and user first_name, you would change the code below to `line[0], line[1]`
        try:
            session.execute(query, (int(line[8]), int(line[3]), line[0], line[9], float(line[5])))
        except Exception as e:
            print(e)
            

#### Do a SELECT to verify that the data have been inserted into each table

In [ ]:
## Add in the SELECT statement to verify the data was entered into the table
query = "SELECT artist, song, length FROM music_session_item WHERE sessionId=338 AND itemInSession=4;"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)

data = [row._asdict() for row in rows]
df = pd.DataFrame(data)
df

### COPY AND REPEAT THE ABOVE THREE CELLS FOR EACH OF THE THREE QUESTIONS

In [ ]:
## Query 2: Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name)\
## for userid = 10, sessionid = 182
query = """
    CREATE TABLE IF NOT EXISTS user_session_songs (
        userId int,
        sessionId int,
        itemInSession int,
        artist text,
        song text,
        firstName text,
        lastName text,
        PRIMARY KEY ((userId, sessionId), itemInSession)
    );
"""
try:
    session.execute(query)
except Exception as e:
    print(e)        
                   

The `user_session_songs` table uses a composite primary key: `PRIMARY KEY ((userId, sessionId), itemInSession)`.

- **Partition Key (`userId`, `sessionId`)**: This groups all song play events for a specific user and session together on the same node, enabling efficient retrieval of all songs played by a user in a particular session.
- **Clustering Key (`itemInSession`)**: This orders the rows within each partition by the order in which the songs were played, allowing for sorting and efficient queries by item sequence.

This design supports queries that need to retrieve all songs played by a user in a session, sorted by the order they were played, along with user and song details.

In [ ]:
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        query = "INSERT INTO user_session_songs (userId, sessionId, itemInSession, artist, song, firstName, lastName) VALUES (%s, %s, %s, %s, %s, %s, %s)"

        try:
            session.execute(query, (int(line[10]), int(line[8]), int(line[3]), line[0], line[9], line[1], line[4]))
        except Exception as e:
            print(e)
            

In [ ]:
## Query 2: Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name)\
## for userid = 10, sessionid = 182
query = "SELECT artist, song, firstName, lastName FROM user_session_songs WHERE userId=10 AND sessionId=182;"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)

data = [row._asdict() for row in rows]
df = pd.DataFrame(data)
df

In [ ]:
## Query 3: Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'
query = """
    CREATE TABLE IF NOT EXISTS song_listeners (
        song text,
        userId int,
        firstName text,
        lastName text,
        PRIMARY KEY (song, userId)
    );
"""
try:
    session.execute(query)
except Exception as e:
    print(e)                    

The `song_listeners` table uses a composite primary key: `PRIMARY KEY (song, userId)`.

- **Partition Key (`song`)**: This allows efficient retrieval of all users who listened to a specific song, as all records for a song are stored together.
- **Clustering Key (`userId`)**: This ensures each user appears only once per song and allows for efficient queries of user details for a given song.

This design directly supports queries like "Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'."
Although some users are found to listen to a song more than once, the query isn't interested in the number of times a user listened to a song, simply the unique users
who listened to the song.

In [ ]:
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        query = "INSERT INTO song_listeners (song, userId, firstName, lastName) VALUES (%s, %s, %s, %s)"

        try:
            session.execute(query, (line[9], int(line[10]), line[1], line[4]))
        except Exception as e:
            print(e)

In [ ]:
# Query 3: Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'
query = "SELECT firstName, lastName FROM song_listeners WHERE song='All Hands Against His Own';"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)

data = [row._asdict() for row in rows]
df = pd.DataFrame(data)
df

### Drop the tables before closing out the sessions

In [ ]:
tables = ['music_session_item', 'user_session_songs', 'song_listeners']
for table in tables:
    query = f"DROP TABLE IF EXISTS {table}"
    try:
        session.execute(query)
    except Exception as e:
        print(e)

### Close the session and cluster connection¶

In [ ]:
session.shutdown()
cluster.shutdown()